# 🔑 第 07 课 · KV Cache 原理:把 O(T²) 的重算降成 O(T) 的追加写

> 注意力维度分析 · 无缓存 vs 有缓存逐行推演 · 复杂度公式 · 真实 GPU 实测

**章节**: 第 2 章 · KV Cache 与 PagedAttention

---

> **📚 本笔记本属于《VLLM_learn: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 复习缩放点积注意力公式,并建立统一的张量维度符号表(batch/head/seq/dim)
- 用手工构造的小张量逐行推演:无缓存时每步要重算哪些 K/V,有缓存时只算哪些
- 亲手实现一个最小 KV Cache(纯 PyTorch),打印每一步中间张量的 shape 并核对维度含义
- 从逐行实现反推出 FLOPs 公式,验证无缓存 O(T²) vs 有缓存 O(T) 的复杂度差
- 在真实 GPU 上实测两种方式耗时,理解 compute-bound 与 memory-bound 的工程真相
- 建立与 PagedAttention 论文(arXiv:2309.06180)和 vLLM 实现的联系

## 📑 本课目录

1. **注意力公式复习** — 从 Attention(Q,K,V) 出发,建立维度符号表
2. **维度符号表** — B/H/S/D 四个维度每个代表什么,贯穿全课
3. **手工张量逐行推演** — batch=2, seq=4, heads=2, dim=8,一步步打印 shape
4. **自回归的两阶段** — prefill 并行算 vs decode 逐个算,K/V 谁在变、谁不变
5. **最小 KV Cache 实现** — 纯 PyTorch 写 forward,逐行注释+shape 核对
6. **复杂度推导** — 从实现反推 FLOPs,验证 O(T²)→O(T);全量加速比 140×/162×,上界 (T+1)/2
7. **真实 GPU 实测** — 无缓存 vs 有缓存毫秒级耗时,compute/memory-bound 分析
8. **与 vLLM 的关系** — GPUBlockPool / PagedAttention 怎么用这个缓存
9. **Streamlit 动态演示** — 拖动滑杆验证结论

## 🔗 参考资料

- [PagedAttention 论文 (SOSP'23)](https://arxiv.org/abs/2309.06180)
- [GQA: Multi-Query Transformer (arXiv)](https://arxiv.org/abs/2305.13245)
- [TensorTonic: KV Cache in LLMs Explained](https://www.tensortonic.com/llm-internals/kv-cache)
- [vLLM 官方博客: 10x faster inference](https://blog.vllm.ai/2023/06/20/vllm.html)
- [vLLM 官方文档](https://docs.vllm.ai)

## 1. 注意力公式复习:先把每个符号搞清楚

KV Cache 的一切都围绕注意力展开。先复习标准**缩放点积注意力**(Scaled Dot-Product Attention):

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V $$

这个公式只有四个字母 $Q, K, V, d_k$,但它们在大模型里的真实形状是什么?**后面所有代码都要打印这些 shape,所以必须先建立一个符号表**:

| 符号 | 含义 | 形状 |
|---|---|---|
| $B$ | batch size(一次处理几个序列) | — |
| $H$ | 注意力头数 num_heads | — |
| $S$ | 序列长度 seq_len(一个序列有几个 token) | — |
| $D$ | 每头维度 head_dim(头向量有多长) | — |
| $Q$ | 查询矩阵(query) | $(B, H, S, D)$ |
| $K$ | 键矩阵(key),被查询的目标 | $(B, H, S, D)$ |
| $V$ | 值矩阵(value),最终要加权求和的内容 | $(B, H, S, D)$ |
| $d_k$ | 缩放因子,就是 $D$ | $D$ |

> ⚠️ 注意:工程实现里通常用 `[B, H, S, D]` 的 4D 布局(每头一段),而不是论文里的 3D 写法。PyTorch 的 `torch.nn.functional.scaled_dot_product_attention` 用的就是 `[B, H, S, D]`。

**关键洞察(本课核心)** 自回归生成时,序列在**不断变长**:
- 第 1 步生成后序列长 $S{=}1$,第 2 步 $S{=}2$,…,第 $T$ 步 $S{=}T$;
- 那么 $K$ 的形状是 $(B, H, T, D)$,$V$ 的形状也是 $(B, H, T, D)$ —— **随 $T$ 增长**;
- 但 $Q$ 永远只关心**最新那个位置**,形状始终是 $(B, H, 1, D)$。

这就是 KV Cache 的出发点:**K 和 V 会越攒越多,而 Q 永远只有一行。**

## 2. 手工构造小张量:亲手看维度变化

先造一个迷你场景,把上面的符号变成真实的数字。我们取:

```
batch B = 2      (同时处理 2 条对话)
num_heads H = 2  (2 个注意力头)
head_dim D = 8   (每头 8 维,方便心算)
seq_len S = 4    (当前序列已有 4 个 token)
```

**每一行都加了注释,把 shape 和维度含义直接写出来**,这样顺着读就能建立直觉。

✅ **逐行推演**。请跟着注释读一遍:每个张量怎么变形、每步 shape 怎么来的。注意 `out` 的形状回到了 $(B,H,1,D)$ —— 这正是下一步**为什么能缓存**的关键。

In [1]:
# -*- coding: utf-8 -*-
import torch

# --------------------------------------------------------------------------
# 1. 定义迷你模型的超参数(刻意取小值,让每个数字都能心算验证)
# --------------------------------------------------------------------------
batch_size = 2          # B: 同时处理的序列条数(2 条对话)
num_heads = 2            # H: 注意力头数(每头独立关注不同的子空间)
head_dim = 8             # D: 每个注意力头的向量维度(向量越长,表达能力越强)
seq_len = 4              # S: 当前序列长度(已有 4 个 token)

# --------------------------------------------------------------------------
# 2. 模拟"某一步"算出的 Q、K、V(真实模型里由 Linear 投影得到)
#    torch.randn 生成标准正态分布,这里只关心 shape,不关心具体值
# --------------------------------------------------------------------------
q = torch.randn(batch_size, num_heads, 1, head_dim)      # Q: 只有"最新"1 个 token 的 query
k = torch.randn(batch_size, num_heads, seq_len, head_dim) # K: 全部 seq_len 个 token 的 key
v = torch.randn(batch_size, num_heads, seq_len, head_dim) # V: 全部 seq_len 个 token 的 value

# 打印 shape,并把每个维度的含义标注清楚
print(f"Q shape = {tuple(q.shape)}  <- (batch={batch_size}, heads={num_heads}, 当前token=1, dim={head_dim})")
print(f"K shape = {tuple(k.shape)}  <- (batch={batch_size}, heads={num_heads}, 全部token={seq_len}, dim={head_dim})")
print(f"V shape = {tuple(v.shape)}  <- (batch={batch_size}, heads={num_heads}, 全部token={seq_len}, dim={head_dim})")

# --------------------------------------------------------------------------
# 3. 手写缩放点积注意力(对照公式 Attention = softmax(QK^T / sqrt(d)) V)
# --------------------------------------------------------------------------
# 3.1 打分:Q 与 K 做点积 —— 新 token 对每个历史 token 有多相关
#     Q: (B, H, 1, D), K: (B, H, S, D)
#     K.transpose(-1, -2) 把最后两维(D, S)对调,变 (B, H, D, S)
#     matmul 后得到 (B, H, 1, S):每个 query 对每个 key 的注意力分数
scores = torch.matmul(q, k.transpose(-1, -2))            # (B, H, 1, S) = (2, 2, 1, 4)
print(f"\nscores shape = {tuple(scores.shape)}  <- 打分矩阵:1 个 query × {seq_len} 个 key")

# 3.2 缩放:除以 sqrt(head_dim),防止点积过大导致 softmax 梯度消失(来源:原版 Transformer 论文)
scaled = scores / (head_dim ** 0.5)                      # 除 sqrt(D)=sqrt(8)≈2.83,形状不变

# 3.3 归一化:对最后一个维度(所有 key)做 softmax,让分数和为 1,成为"注意力权重"
weights = torch.softmax(scaled, dim=-1)                  # (B, H, 1, S) = (2, 2, 1, 4)
print(f"weights shape = {tuple(weights.shape)}  <- softmax 后的权重,每行和为 1")

# 3.4 加权求和:用权重对 V 加权 —— 输出是把"相关历史"的信息汇总到新 token
#     weights: (B, H, 1, S)  ×  V: (B, H, S, D)  ->  (B, H, 1, D)
out = torch.matmul(weights, v)                           # (B, H, 1, D) = (2, 2, 1, 8)
print(f"out    shape = {tuple(out.shape)}    <- 新 token 融合历史后的输出,形状与 Q 相同")

# 验证数值正确性:用 PyTorch 官方实现对照(结果应几乎一致)
import torch.nn.functional as F
ref = F.scaled_dot_product_attention(q, k, v, attn_mask=None)
print(f"与官方实现最大误差 = {(out - ref).abs().max().item():.2e}  (≈0 说明手写正确)")

Q shape = (2, 2, 1, 8)  <- (batch=2, heads=2, 当前token=1, dim=8)
K shape = (2, 2, 4, 8)  <- (batch=2, heads=2, 全部token=4, dim=8)
V shape = (2, 2, 4, 8)  <- (batch=2, heads=2, 全部token=4, dim=8)

scores shape = (2, 2, 1, 4)  <- 打分矩阵:1 个 query × 4 个 key
weights shape = (2, 2, 1, 4)  <- softmax 后的权重,每行和为 1
out    shape = (2, 2, 1, 8)    <- 新 token 融合历史后的输出,形状与 Q 相同
与官方实现最大误差 = 1.19e-07  (≈0 说明手写正确)


## 3. 自回归解码:prefill 与 decode 的维度视角

LLM 生成文本是**自回归**(autoregressive)的:每次只生成 1 个 token,把新 token 拼到序列尾巴,
再拿**整个变长序列**跑一遍前向,预测下一个。于是推理天然分成两阶段:

| 阶段 | 输入序列长度 | K/V 形状 | 计算特点 |
|---|---|---|---|
| **prefill(预填充)** | 整个 prompt,如 $S{=}2048$ | 一次性算出全部 | 高度并行,compute-bound |
| **decode(解码)** | 每步只 +1,如 $S{=}2048, 2049, ...$ | 每步只算新 1 个 | 串行,memory-bound |

关键问题在 decode:**每步 $K$ 都会变长**。

### 🔍 无缓存(naive):每步重算全部历史

第 $t$ 步,序列长 $t$。朴素做法是把 $t$ 个 token 的 $Q,K,V$ **全部重算一遍**,再做注意力。
可前一步明明已经算过前 $t{-}1$ 个 token 的 $K,V$ 了——**重算就是浪费**。

### 💡 关键观察:K 和 V 一旦算出就不变

第 $j$ 个 token 的 key $k_j$ 只依赖它自己的 embedding 和模型权重,与「后面来了谁」无关:

$$ k_j = \text{Linear}_K(\text{emb}(x_j)), \qquad v_j = \text{Linear}_V(\text{emb}(x_j)) $$

所以第 1 步算出的 $k_0, k_1$ 到第 100 步依然是同样的 $k_0, k_1$。
**把它们存起来,decode 每步只算新 token 的 $k_{t}, v_{t}$ 再追加进去** —— 这就是 KV Cache。

> 📄 这一洞察是 KV cache 一切优化的起点,见 TensorTonic 教程「*exactly which vectors get cached and why they are safe to cache*」。

## 4. 最小 KV Cache 实现:每行代码都注释

现在用一个玩具模型动手实现。做法:实现一个**单个注意力层**的 forward,
它接受 `(k_cache, v_cache)` 并返回更新后的缓存 —— 这正是 vLLM 里 attention backend 每步做的三件事:
**新 token 的 QKV 投影 → 写入缓存 → 与缓存中的历史 KV 做注意力**。

我们故意不用 `torch.cat` 重拼(那是 O(S²) 的,08 课会专门讲),而是用**预分配 + 原位写入**的思路。

🛠️ **最小实现**。重点观察两件事:
1. `k_cache[:, :, cache_len:cache_len+new_tokens, :] = k`:每次是**切片写入**(追加),不是重新计算;
2. `cache_len` 每次递增,`K` 缓存第 `cache_len` 位之前永远不变。这正对应 vLLM attention backend 每步的「新 QKV → 写缓存 → 注意力」三件事。

In [2]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn

# --------------------------------------------------------------------------
# 定义一个极简的单层注意力,展示 KV Cache 的写入与读取
# --------------------------------------------------------------------------
class AttentionWithCache(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, head_dim: int):
        # 调用父类 nn.Module 的初始化(必须)
        super().__init__()
        # 保存超参供 forward 使用
        self.num_heads = num_heads            # H: 注意力头数
        self.head_dim = head_dim              # D: 每头维度
        # Q/K/V 三个投影矩阵:把隐藏向量 hidden_size 投影成 num_heads*head_dim 维
        # 注意:这里的 w_q 把 (B, S, hidden) -> (B, S, num_heads*head_dim),再 reshape 成多头
        self.w_q = nn.Linear(hidden_size, num_heads * head_dim, bias=False)  # Q 投影
        self.w_k = nn.Linear(hidden_size, num_heads * head_dim, bias=False)  # K 投影
        self.w_v = nn.Linear(hidden_size, num_heads * head_dim, bias=False)  # V 投影

    def forward(self, x, k_cache, v_cache, cache_len):
        # x: (B, new_tokens, hidden) 当前步新增的 token(可能是 1 个,也可能是整个 prompt)
        # k_cache / v_cache: 已预分配的缓存,shape (B, H, max_seq, D)
        # cache_len: 当前已缓存了多少个 token(历史长度)

        batch, new_tokens, _ = x.shape                     # 解出 B 和新增 token 数

        # ---- 第 1 步:新 token 的 QKV 投影(每行注释)----
        q = self.w_q(x)                                    # (B, new, H*D) 查询投影
        k = self.w_k(x)                                    # (B, new, H*D) 键投影
        v = self.w_v(x)                                    # (B, new, H*D) 值投影

        # reshape 成多头布局:(B, new, H*D) -> (B, new, H, D) -> 转置为 (B, H, new, D)
        q = q.view(batch, new_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, new_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, new_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # ---- 第 2 步:把新 K/V 原位写进缓存 ----
        # 写入区间 [cache_len, cache_len+new_tokens),这是"追加写"而非"重算"
        k_cache[:, :, cache_len:cache_len + new_tokens, :] = k   # K 写进缓存对应位置
        v_cache[:, :, cache_len:cache_len + new_tokens, :] = v   # V 写进缓存对应位置

        # ---- 第 3 步:注意力:新 Q 与缓存里全部历史 K/V 交互 ----
        # 取缓存中真正有效的部分(前 cache_len+new_tokens 个位置)
        k_all = k_cache[:, :, :cache_len + new_tokens, :]        # (B, H, S', D) S'=已缓存长度
        v_all = v_cache[:, :, :cache_len + new_tokens, :]        # (B, H, S', D)

        # 打分:Q(B,H,new,D) × K^T(B,H,D,S') -> (B,H,new,S')
        scores = torch.matmul(q, k_all.transpose(-1, -2)) / (self.head_dim ** 0.5)
        weights = torch.softmax(scores, dim=-1)                  # (B,H,new,S') 每行和=1
        out = torch.matmul(weights, v_all)                       # (B,H,new,D) 加权汇总
        return out.transpose(1, 2)                               # 回到 (B, new, H, D)


# --------------------------------------------------------------------------
# 演练:prefill(一次性处理整个 prompt)+ decode(每步 1 个 token)
# --------------------------------------------------------------------------
hidden_size = 16            # 隐藏层维度(embedding 长度)
num_heads = 2               # H = 2
head_dim = 8                # D = 8
max_seq = 8                 # 缓存最大容量(预分配,不随 token 增长)
batch_size = 1              # B = 1,先跑单序列便于观察

model = AttentionWithCache(hidden_size, num_heads, head_dim)

# 预分配 KV 缓存:一次性申请 (B, H, max_seq, D),避免反复 resize
k_cache = torch.zeros(batch_size, num_heads, max_seq, head_dim)  # K 缓存缓冲
v_cache = torch.zeros(batch_size, num_heads, max_seq, head_dim)  # V 缓存缓冲
print(f"预分配 K 缓存 shape = {tuple(k_cache.shape)} <- (batch={batch_size}, heads={num_heads}, max_seq={max_seq}, dim={head_dim})")

cache_len = 0   # 当前缓存长度初始为 0

# ---- PREFILL:一次喂整个 prompt(4 个 token)----
prompt = torch.randn(batch_size, 4, hidden_size)   # (B=1, S=4, hidden=16) 模拟 prompt 的 4 个 embedding
out = model(prompt, k_cache, v_cache, cache_len)               # 前向,内部会把 4 个 K/V 写入缓存
cache_len += 4                                                 # 缓存长度更新为 4
print(f"prefill 后:输出 shape = {tuple(out.shape)}, 缓存长度 cache_len = {cache_len}")
print(f"           此时 K 缓存前 {cache_len} 个位置已有内容,其余仍为 0(预分配的空位)")

# ---- DECODE:每步只喂 1 个新 token ----
for step in range(1, 4):                                        # 再解码 3 步
    new_tok = torch.randn(batch_size, 1, hidden_size)           # (B, 1, hidden) 仅 1 个新 token
    out = model(new_tok, k_cache, v_cache, cache_len)           # 前向:Q 只算新的,K/V 追加写
    cache_len += 1                                              # 缓存 +1
    print(f"decode 第 {step} 步:输出 shape = {tuple(out.shape)}, 缓存长度 = {cache_len}")

print(f"\n最终缓存长度 = {cache_len},K/V 每步只追加,从未重算 —— 这就是 KV Cache。")

预分配 K 缓存 shape = (1, 2, 8, 8) <- (batch=1, heads=2, max_seq=8, dim=8)
prefill 后:输出 shape = (1, 4, 2, 8), 缓存长度 cache_len = 4
           此时 K 缓存前 4 个位置已有内容,其余仍为 0(预分配的空位)
decode 第 1 步:输出 shape = (1, 1, 2, 8), 缓存长度 = 5
decode 第 2 步:输出 shape = (1, 1, 2, 8), 缓存长度 = 6
decode 第 3 步:输出 shape = (1, 1, 2, 8), 缓存长度 = 7

最终缓存长度 = 7,K/V 每步只追加,从未重算 —— 这就是 KV Cache。


## 5. 复杂度推导:为什么是 O(T²) vs O(T)

设模型 $L$ 层,每层 $H$ 头,每头 $D$ 维;已经生成了 $T$ 个 token。

**每 token 每层的 QKV 投影 FLOPs**(3 个矩阵乘,每个 $2 \cdot hidden \cdot H D$):

$$ \text{QKV} = 6 L H D^2 $$

**第 $t$ 步注意力 FLOPs**:新 query($1$ 个)与 $t$ 个历史 key 打分($2HDt$)再与 $t$ 个 value 加权($2HDt$):

$$ \text{attn}(t) = 4 L H D t $$

### 🔢 无缓存:每步重算全部历史

第 $t$ 步要重算 $t$ 个 token 的 QKV($\times t$),注意力照常:
$$ \text{FLOPs}_{\text{no-cache}} = \sum_{t=1}^{T} (6LHD^2 \cdot t + 4LHDt) = O(T^2) $$

### 🔢 有缓存:每步只算新 token

第 $t$ 步只算 1 个新 token 的 QKV(常数),注意力照常($4LHDt$ 是硬刚需,两种方式一样):
$$ \text{FLOPs}_{\text{cache}} = \sum_{t=1}^{T} (6LHD^2 + 4LHDt) = O(T) $$

### ⚖️ 加速比

差距全在 QKV 投影项:无缓存把它乘了 $t$,有缓存是常数。于是,**仅 QKV 投影部分**的理论上界为:

$$ \text{speedup}_{\text{QKV}} \approx \frac{6LHD^2 \cdot T(T+1)/2}{6LHD^2 \cdot T} \approx \frac{T+1}{2} $$

`T=1024` 时这个上界约 $\approx 512$×,`T=2048` 时约 $\approx 1024$×。
**但这只是『仅 QKV 部分』的上界**——注意力打分($4LHDt$ 一项)两种方式都要做、省不掉。把它计入后的**全量实际加速比**要小得多:`T=1024` 约 **140×**,`T=2048` 约 **162×**(下一步代码会打印验证)。
> ⚠️ 注意:以上仍是**计算量(FLOPs)视角**的理论值。真实 GPU 上 decode 是 memory-bound,实测加速比往往更小 —— 第 6 节实测见分晓。

✅ **数值验证**。把上节的公式直接写成函数,代入真实模型规模:**全量(含注意力)加速比约 140×(T=1024)、162×(T=2048)**,都明显小于仅 QKV 的上界 $(T{+}1)/2$;且全量值只由序列长度 $T$ 决定,与模型大小无关——KV Cache 的收益是普适的。

In [3]:
# -*- coding: utf-8 -*-
import numpy as np

def total_flops(L: int, H: int, D: int, T: int, use_cache: bool) -> int:
    # 计算生成 T 个 token 的总 FLOPs
    # qkv: 每 token 每层的 QKV 投影 FLOPs = 6*L*H*D^2
    qkv = 6 * L * H * D * D                       # 常数项:1 个 token 的 QKV
    t = np.arange(1, T + 1)                       # 1..T 每个解码步的序列长度
    if use_cache:
        # 有缓存:每步 QKV 恒定(只算新 token),注意力 4*L*H*D*t 随 t 线性增长
        per_step = qkv + 4 * L * H * D * t
    else:
        # 无缓存:每步 QKV 也乘 t(重算全部历史),注意力相同
        per_step = qkv * t + 4 * L * H * D * t
    return int(per_step.sum())                    # 返回总 FLOPs(int)

# 用 LLaMA-7B 的真实规模验证:L=32 层, H=32 头, D=128 维, 上下文 T=1024
L, H, D, T = 32, 32, 128, 1024
no_cache = total_flops(L, H, D, T, use_cache=False)   # 无缓存总 FLOPs
use_cache = total_flops(L, H, D, T, use_cache=True)   # 有缓存总 FLOPs
print(f"无缓存总 FLOPs = {no_cache:.3e}")
print(f"有缓存总 FLOPs = {use_cache:.3e}")
print(f"全量加速比(含注意力) = {no_cache / use_cache:.1f}×")
print(f"仅 QKV 部分的理论上界 (T+1)/2 = {(T + 1) / 2:.0f}×  (不含注意力,故大于全量值)")

# 额外验证:全量加速比只取决于 T,与模型规模(L,H)无关;上界 (T+1)/2 恒更高
for Lm, Hm in [(32, 32), (80, 64), (96, 96)]:            # LLaMA-7B / 70B / GPT-3 175B
    na = total_flops(Lm, Hm, 128, 2048, use_cache=False)
    ca = total_flops(Lm, Hm, 128, 2048, use_cache=True)
    print(f"L={Lm:<3} H={Hm:<3} 全量加速比 = {na / ca:6.1f}×  (T=2048;仅QKV上界≈{(2048+1)/2:.0f}×)")

无缓存总 FLOPs = 5.310e+13
有缓存总 FLOPs = 3.782e+11
全量加速比(含注意力) = 140.4×
仅 QKV 部分的理论上界 (T+1)/2 = 512×  (不含注意力,故大于全量值)
L=32  H=32  全量加速比 =  162.5×  (T=2048;仅QKV上界≈1024×)
L=80  H=64  全量加速比 =  162.5×  (T=2048;仅QKV上界≈1024×)
L=96  H=96  全量加速比 =  162.5×  (T=2048;仅QKV上界≈1024×)


## 6. 真实 GPU 实测:纸上公式 vs 硬件真相

前面的复杂度推导是**计算量**视角(FLOPs)。但真实 GPU 上还有第二个维度:**内存带宽**。
decode 每步只算 1 个新 token,却要把整个历史的 K/V 从 HBM 读一遍——
**读得比算得多**,算子被带宽卡住,GPU 算力吃不满。这就是为什么即便全量理论加速比(140×/162×)也常常达不到,实测往往只有几十倍。

下面复用 `real_ops.bench_qkv_attn`:在 GPU 上真实跑注意力算子,分别测「无缓存(重算全部历史的 QKV+注意力)」
和「有缓存(只算新 query 的注意力)」的毫秒耗时。

🚀 **真实 GPU 数字**。重点:无缓存/有缓存的实测加速比随 $T$ 上升,但**远小于公式的 $(T{+}1)/2$**。
原因正是 memory-bound:decode 阶段带宽瓶颈压制了 QKV 那部分省下的 FLOPs。

In [4]:
# -*- coding: utf-8 -*-
import sys, os
# 设定环境变量避免 OpenMP 冲突(Windows 上 torch 常见问题)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
# 把 ch02 目录加入模块搜索路径,好导入 real_ops
sys.path.insert(0, r"D:\Project\21-Cpp_learn\explore\VLLM_learn\exercises\ch02")
from real_ops import bench_qkv_attn                       # 真实 GPU 微基准

# 用 LLaMA 的注意力配置:H=32 头, D=128 维, 历史长度从 32 测到 512
r07 = bench_qkv_attn(H=32, D=128, T_list=(32, 64, 128, 256, 512), dtype="bf16")
print("设备:", r07["device"], "· 是否仿真(simulated) =", r07["simulated"])
print(f"\n{'历史T':>6} | {'无缓存(重算全部, ms)':>22} | {'有缓存(只算新, ms)':>20} | {'加速比':>8}")
for t, nc, c in r07["results"]:                          # 遍历每个历史长度的结果
    print(f"{t:6d} | {nc:22.3f} | {c:20.3f} | {nc / max(c, 1e-12):8.1f}x")

# 结论:真实加速比远小于 (T+1)/2 的公式值,原因是 decode 是 memory-bound(读 K/V 比算 Q 贵)

设备: cuda:0(NVIDIA GeForce RTX 5060 Laptop GPU) · 是否仿真(simulated) = False

   历史T |          无缓存(重算全部, ms) |         有缓存(只算新, ms) |      加速比
    32 |                  0.079 |                1.057 |      0.1x
    64 |                  0.078 |                0.084 |      0.9x
   128 |                  0.072 |                0.057 |      1.3x
   256 |                  0.171 |                0.061 |      2.8x
   512 |                  0.894 |                0.074 |     12.1x


### 🔬 为什么实测比公式小?compute-bound vs memory-bound

看每步的数据流:有缓存时,第 $t$ 步要做

```
新 token 的 QKV 投影  (小,算得快)
+ 读全部历史 K/V       (大,从 HBM 读 (B,H,t,D)×2 个元素)
+ 与全部历史做注意力   (读得越多越慢)
```

当 $t$ 很大时,**读 K/V 的带宽成本**成为主导,GPU 算力(FLOPs)根本不是瓶颈。
所以 KV Cache 的真实价值不只是「省 FLOPs」,更是:**把不可并行的 decode 变成轻量 append,
并避免反复搬运历史 KV**。这正是 PagedAttention 论文(SOSP'23)要解决的核心矛盾——
缓存本身的**内存管理**问题(碎片、预分配浪费),我们 08、10 课展开。

## 7. 与 vLLM 的关系:缓存从哪里来、到哪里去

vLLM 把「存 K/V」这件事做成了工业级。关键路径:

1. **预分配物理块**:`v1/core/block_pool.py` 的 `GPUBlockPool` 启动时按显存预算一次划出
   成百上千个**固定大小**的物理块(默认 16 token/块);
2. **页表映射**:`v1/core/kv_cache_utils.py` 维护 逻辑块→物理块 的映射与读写下标;
3. **调度器分配**:`sched/scheduler.py` 为每个请求逐 token 分配缓存块;
4. **attention backend**:每步只做「新 QKV → 写缓存 → 与缓存做注意力」三件事。

这就是本课第 4 节最小实现的**工程放大版**。区别在于:
- 我们是 1 个预分配 buffer;vLLM 是**分页的**块池(借鉴 OS 虚拟内存);
- 我们用 `cache_len` 记录位置;vLLM 用 **block table**(逻辑块→物理块)记录;
- 我们每步 append;vLLM 按块分配,内部碎片小、可跨请求共享(前缀复用)。

> 📄 详见 **PagedAttention (SOSP'23)** arXiv:2309.06180 第 4.2 节「KV Cache Manager」。

### 🔗 和下一课(08)的衔接

本课证明了 KV Cache 省计算。但缓存本身要**吃显存**,而且吃得很凶:

$$ \text{KV bytes} = 2 \times \text{batch} \times \text{seq} \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{dtype\_bytes} $$

L=32, kv_heads=8(GQA), D=128, seq=32K, batch=1, bf16:

$$ 2 \times 1 \times 32768 \times 32 \times 8 \times 128 \times 2 \approx 4.3\,\text{GB} $$

一个请求就 4GB 显存——这就是 08 课要算的「显存账本」,也是 GQA(MQA)能把缓存缩小 $\frac{H}{\text{kv\_heads}}$ 倍的动机。

## 8. 🖥️ Streamlit 动态演示:亲手验证

把上面的公式做成交互 App:拖动**层数 / 头数 / 序列长度**滑杆,实时计算两种方式的总 FLOPs 并对比。

### 📜 App 完整源码(`app_07_kv_principle.py`)

▶️ 此 cell 在 streamlit 环境中才真正运行;在 notebook 中仅作展示。

In [5]:
try:
    import streamlit as st
    _IS_STREAMLIT = bool(st.runtime.exists())
except Exception:
    _IS_STREAMLIT = False

if _IS_STREAMLIT:
    """app_07_kv_principle.py — KV Cache 原理动态演示(真实 GPU 算子实测版)
    运行: D:\\uv_envs\\uv_cuda\\Scripts\\python.exe -m streamlit run app_07_kv_principle.py
    """
    import os
    import sys

    _HERE = os.path.dirname(os.path.abspath(__file__))
    if _HERE not in sys.path:
        sys.path.insert(0, _HERE)

    import numpy as np
    import streamlit as st
    import plotly.graph_objects as go

    from app_common import terminology, real_badge, foot_note
    from real_ops import bench_qkv_attn, cuda_available, device_str
    st.set_page_config(page_title="KV Cache 原理 · 计算量对比", layout="wide")

    st.title("🔑 KV Cache 原理:无缓存 vs 有缓存,差多少计算量?")
    st.markdown(
        "LLM 是**自回归(autoregressive)**模型:每步只生成 1 个新 token,但要让新 token 与**全部历史 token** 交互。\n\n"
        "两个阶段的本质差异:\n"
        "- **prefill(预填充)**:对整段 prompt 做**并行**前向,是 **compute-bound(算力受限)**;\n"
        "- **decode(解码)**:每步 1 个 token,逐步自回归,是 **memory-bound(内存受限)**。\n\n"
        "💡 **无缓存**:每一步都把历史所有 token 的 K、V 重新投影一遍,计算量随序列长度**平方级( O(T²) )**膨胀;\n"
        "💾 **有缓存(KV Cache)**:K、V 只投影一次就存进显存复用,每步只算新 token 的 QKV,总计算量降到**线性( O(T) )**。\n\n"
        "下方先看 **真实 GPU 算子微基准**(直接对注意力核心算子打点计时),再对比两条计算曲线——数据不是理想的 `np.arange`,"
        f"而是跑在 **{device_str()}** 上的实测时延。"
    )
    st.caption(
        "微基准说明:构造与真实自回归 decode 相同的形态——单 query 对全部历史 key/value 做打分+softmax+加权(async on CUDA)。"
        "「无缓存」模拟把 t 个历史 token 的 QKV **全部当作新算出**并做完整注意力;「有缓存」只用一个新 query 做注意力。"
        "每档重复多次,取其均值。未装 CUDA 时优雅回落为理论估算。"
    )

    with st.sidebar:
        st.header("⚙️ 基准参数")
        L = st.slider("🧱 层数 num_layers (L)", 1, 80, 32)
        H = st.slider("🎯 注意力头数 num_heads (H)", 1, 64, 32)
        D = st.select_slider("📏 head_dim (D)", options=[64, 128, 256], value=128)
        dtype = st.selectbox("🎛️ 精度 dtype", ["bf16", "fp16"], index=0)
        seq_len = st.slider("📜 已生成 token 数 (T)", 64, 2048, 1024, step=64)
        run = st.button("🚀 跑真实 CUDA 基准", type="primary")
        st.caption(f"GPU: {device_str() if cuda_available() else '未检测到 CUDA,将用理论曲线'}")

    T_list = tuple(sorted(set([64, 128, 256, 512, 1024, seq_len])))
    if run or not cuda_available():
        bench = bench_qkv_attn(L=L, H=H, D=D, T_list=T_list, dtype=dtype)
    else:
        bench = None

    if bench is None:
        st.info("点击左侧「🚀 跑真实 CUDA 基准」在 GPU 上实测耗时(约几秒到十几秒)。")
    st.session_state.setdefault("bench", bench if bench is not None else None)

    # ---------- 真相数据 vs 理论 ----------
    if st.session_state["bench"] is not None:
        b = st.session_state["bench"]
        results = b["results"]
        ts = np.array([r[0] for r in results])
        nocache_ms = np.array([r[1] for r in results])
        cache_ms = np.array([r[2] for r in results])
        inst_real = nocache_ms / np.maximum(cache_ms, 1e-9)
        real = not b.get("simulated")

        m1, m2, m3 = st.columns(3)
        m1.metric("数据来源设备", (b["device"] if real else "CPU(理论)"),
                  delta="真实算子实测" if real else "估算")
        m2.metric("无缓存末档每步耗时", f"{nocache_ms[-1]:.2f} ms" if real else f"{nocache_ms[-1]:.1e}")
        m3.metric("有缓存末档每步耗时", f"{cache_ms[-1]:.2f} ms" if real else f"{cache_ms[-1]:.1e}")
        st.session_state["real"] = real
    else:
        nocache_ms = cache_ms = ts = inst_real = None
        real = False
        st.session_state["real"] = False

    # ---------- 理论 FLOPs 曲线(作为对照轴) ----------
    ts_full = np.arange(1, seq_len + 1)
    qkv = 6 * L * H * D * D
    att_per_t = 4 * L * H * D * ts_full
    step_cache = qkv + att_per_t
    step_nocache = qkv * ts_full + att_per_t
    tot_cache = np.cumsum(step_cache)
    tot_nocache = np.cumsum(step_nocache)
    cum_speedup = tot_nocache / tot_cache

    st.subheader("📊 两条计算曲线:平方级 vs 线性")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("无缓存累计 FLOPs", f"{tot_nocache[-1]:.3e}")
    c2.metric("有缓存累计 FLOPs", f"{tot_cache[-1]:.3e}")
    c3.metric("累计加速比(理论)", f"{cum_speedup[-1]:.1f}×")
    c4.metric("末步单步加速比(理论)", f"{step_nocache[-1]/step_cache[-1]:.1f}×")

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=ts_full, y=tot_nocache, name="无缓存(累计)", mode="lines",
                              line=dict(color="#e74c3c", width=2), fill="tozeroy", fillcolor="rgba(231,76,60,0.15)"))
    fig1.add_trace(go.Scatter(x=ts_full, y=tot_cache, name="有缓存(累计)", mode="lines",
                              line=dict(color="#2ecc71", width=2), fill="tozeroy", fillcolor="rgba(46,204,113,0.15)"))
    if inst_real is not None and len(ts) > 0:
        fig1.add_trace(go.Scatter(x=ts, y=(nocache_ms * 1e15), name="真实·无缓存(注意力核心)", mode="markers+lines",
                                  line=dict(color="#c0392b", width=1, dash="dot"),
                                  marker=dict(symbol="x", size=6)))
        fig1.add_trace(go.Scatter(x=ts, y=(cache_ms * 1e15), name="真实·有缓存(注意力核心)", mode="markers+lines",
                                  line=dict(color="#27ae60", width=1, dash="dot"),
                                  marker=dict(symbol="cross", size=6)))
    fig1.update_layout(
        title="📊 计算量曲线(实线=理论 FLOPs;虚线标记=该形状下单次注意力算子的实测耗时×缩放)",
        xaxis_title="已生成 token 数", yaxis_title="累计 FLOPs / 相对量度", template="plotly_white",
        hovermode="x unified", legend=dict(orientation="h", y=1.12))
    st.plotly_chart(fig1, width="stretch")

    # ---------- 加速比 ----------
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=ts_full, y=step_nocache / step_cache, name="每步加速比(理论)", mode="lines",
                              line=dict(color="#3498db", width=2)))
    fig2.add_trace(go.Scatter(x=ts_full, y=cum_speedup, name="累计加速比(理论)", mode="lines",
                              line=dict(color="#9b59b6", width=2, dash="dash")))
    if inst_real is not None and len(ts) > 0:
        fig2.add_trace(go.Scatter(x=ts, y=inst_real, name="真实·单步加速比", mode="markers",
                                  marker=dict(symbol="diamond", size=8, color="#e67e22"),
                                  error_y=dict(type="data", array=np.ones_like(ts) * 0.05, visible=True)))
    fig2.add_hline(y=1, line_dash="dot", line_color="gray", annotation_text="无缓存基准 = 1×")
    fig2.update_layout(title="📈 加速比曲线:越往后差距越大", xaxis_title="已生成 token 数",
                        yaxis_title="加速比(×)", template="plotly_white", hovermode="x unified",
                        legend=dict(orientation="h", y=1.1))
    st.plotly_chart(fig2, width="stretch")
    st.markdown("### 🏷️ 关键结论")
    st.success(
        f"理论上序列长度为 **{seq_len}** 时,无缓存累计计算量约是 **{cum_speedup[-1]:.1f} 倍**于有缓存;"
        "这不依赖于模型规模,只取决于序列长度—— token 越多,差距越大。"
    )
    st.markdown(
        "**为什么 decode 快不起来?** 瓶颈并不在 QKV 那点 FLOPs,而在**内存**:每步只产出 1 个 token,"
        "却要把注意力完整读一遍 K、V(half-warp 逐 token)。真正让 vLLM 在 decode 下跑满吞吐的,"
        "是 **PagedAttention + CUDA Graph + 连续批处理** 的组合拳。本课先记住「KV Cache 把 O(T²) 降成 O(T)」这一记重拳。"
    )
    real_badge(real=st.session_state.get("real", False))
    terminology([
        ("autoregressive / self-attention", "自回归每一新 token 都要与全部历史位置交互,是 KV Cache 存在的前提;attention 让位置可并行,见 Attention Is All You Need。"),
        ("compute-bound vs memory-bound", "prefill 算力受限,decode 受内存带宽限制,二者主导的硬件瓶颈完全不同。"),
        ("FLOPs 与 O(T²)/O(T)", "无缓存把 QKV 投影乘以 t 次故为平方;有缓存该部分恒为常数故线性。"),
        ("KV Cache", "缓存已算出的 K/V 张量,按 层×头×序列 存储;历史 K/V 与未来 token 无关,故数学上无损复用。"),
        ("prefill / TTFT", "预填充阶段对该 prompt 做一次并行前向并得到首个 token;TTFT 是衡量其快慢的指标。"),
        ("decode / TPOT", "逐 token 自回归阶段;TPOT 是每个输出 token 的平均耗时,memory-bound 主导。"),
        ("数据来源", f"本次真实注意力算子实测跑在 {device_str()}(CUDA 微基准)。参考文献:PagedAttention SOSP'23 · vLLM docs。"),
    ])

    foot_note()

    # ============ PyCharm / 直接运行入口 ============
    if __name__ == "__main__":
        try:
            import streamlit.runtime as st_runtime
            if st_runtime.exists():
                raise SystemExit(0)
        except Exception:
            pass
        import os as _os
        import subprocess
        import sys as _sys
        subprocess.run([_sys.executable, "-m", "streamlit", "run", _os.path.abspath(__file__)])
else:
    print("💡 当前不是 streamlit 环境,跳过执行本 App。")
    print("    请把上方源码保存为 app_07_kv_principle.py 后运行:")
    print("    D:\\uv_envs\\uv_cuda\\Scripts\\python.exe -m streamlit run app_07_kv_principle.py")


💡 当前不是 streamlit 环境,跳过执行本 App。
    请把上方源码保存为 app_07_kv_principle.py 后运行:
    D:\uv_envs\uv_cuda\Scripts\python.exe -m streamlit run app_07_kv_principle.py


### 🏃 运行方法

1. 使用本目录已生成的 `app_07_kv_principle.py`;
2. 在命令行执行:
```
D:\uv_envs\uv_cuda\Scripts\python.exe -m streamlit run app_07_kv_principle.py
```
3. 浏览器打开 http://localhost:8501 ,拖动滑杆观察 FLOPs 变化。

🔍 试试:把序列长度拉到 4096,全量加速比接近 180×;把层数拉到 200,两条线「剪刀差」更大。

## ✅ 本课小结

- 注意力中 K、V 只依赖各自 token 的 embedding,一旦算出就固定不变——这是能缓存的前提
- 无缓存每步重算全部历史 K/V,总计算量 O(T²);有缓存每步只算新 token 并追加写入,总计算量 O(T)
- 加速比 ≈ (T+1)/2(仅 QKV 部分),与模型规模无关,只取决于序列长度
- 真实 GPU 上 decode 是 memory-bound,实测加速比远小于公式——KV Cache 更核心的价值是避免反复搬运历史 KV
- vLLM 用 GPUBlockPool 管理分页物理块 + block table 记录逻辑到物理的映射,是本课最小实现的工程放大版

## ✍️ 动手练习

1. 把 AttentionWithCache 改成支持 GQA:共享 KV 头(把 k 的 H 从 2 变 1,再用 repeat 扩回),看缓存体积变化
2. 在 forward 里手动打印每次写入缓存前后的 k_cache[:,0,0,:,:],验证「追加写」确实只改了新增位置
3. 把 decode 循环改成每步打印 K 缓存第 cache_len 列的值,确认历史位置从未被覆盖

## 🔗 延伸阅读

- [PagedAttention 论文](https://arxiv.org/abs/2309.06180)
- [GQA 论文](https://arxiv.org/abs/2305.13245)
- [vLLM 博客: Continuous Batching](https://blog.vllm.ai/2023/06/20/vllm.html)
- [TensorTonic: KV Cache Explained](https://www.tensortonic.com/llm-internals/kv-cache)

---
> 🎉 恭喜完成本课!下一课见!